# 02 · LoRA Adaptation Experiments

**Does low-rank adaptation recover full-fine-tune quality at <5% of the parameters —
and forget less — when adapting a recurrent separator to a shifted domain?** This
notebook runs Direction 05 end to end: the pre-registration recap, the domain build,
the G1 sanity + LR probes, the 12 fine-tune runs, the one consolidated test session,
and the analysis (the quality-vs-trainable-params curve, the forgetting table, rank
sensitivity, and the wall-clock/VRAM table). All GPU/data cells are **⚠️ RUN THIS
LATER**; the CPU cells (dry-run, sanity table) run now against the mock host.

Runtimes (MASTER_PLAN §6, T4): LR probes ~1.5–2 h; T1 runs 8×6k ~8–12 h; T2 runs
4×6k ~4–6 h; consolidated eval ~1–1.5 h; domain materialization ~1 h CPU. Ceiling
≈ **18–24 T4-hours**.

### State of this notebook
- **Contract:** [`../MASTER_PLAN.md`](../MASTER_PLAN.md) §2 (hypotheses), §3 (design),
  §5 (protocol), §6 (run book), §7 (gates), §12 (interpretation matrix);
  [`../THEORY.md`](../THEORY.md) §6 (statistics).
- **Prerequisite:** Direction 01 data prep (stereo shards) + `scripts/make_domains.py`.
- **Nothing is trained here.** Every launch cell is RUN LATER and resumable
  (`run_sweep.py` skips completed rows). All decisions freeze on validation before the
  single test pass (gate G3).

In [ ]:
# === Colab bootstrap — RUN THIS LATER (Colab only) ==========================
# ⚠️ RUN THIS LATER · ~2 min · CPU. Installs the pinned env and mounts Drive.
# Skip locally if you already `pip install -r requirements.txt`.
#
# !git clone https://github.com/SeanSalvador2/vocal-separation-research.git
# %cd vocal-separation-research
# !pip install -r requirements.txt
# from google.colab import drive; drive.mount('/content/drive')
# import os; os.environ["SHARD_ROOT"] = "/content/drive/MyDrive/musdb_shards"
print("Bootstrap cell — run on Colab only (see comments). No-op here.")

## 1 · Pre-registration recap (verbatim from MASTER_PLAN §2)

Notation: for domain D and recipe R, let $g_D(R)$ = mean vocals SI-SDR on D's
**target-domain test tracks** minus zero-shot's score on the same tracks
("adapted-domain gain"), and $f(R)$ = zero-shot SI-SDR on the **standard** test set
minus R's SI-SDR on the standard test set after adapting ("forgetting"; positive =
regression). σ_seed = between-seed std pooled over the two 3-seed cells (LoRA-16 and
full-FT on T1).

> **H-05a (efficiency).** On the primary domain T1:
> $g_{T1}(\text{LoRA-16}) \ge 0.9 \cdot g_{T1}(\text{full-FT})$, with LoRA-16 training
> < 5 % of the host's parameters.
> - **Precondition:** $g_{T1}(\text{full-FT}) > \max(2\sigma_{\text{seed}}, 0.3\text{ dB})$
>   — the domain shift must be big enough that adaptation does something; otherwise
>   H-05a is **not evaluable** (pre-registered branch: "the shift was too mild"), and
>   the secondary domain T2 becomes primary for this hypothesis.
> - **Supported / refuted** by the 3-seed means with a seed-propagated CI on the gain
>   ratio (delta method in THEORY §6); ratio ≥ 0.9 with CI excluding < 0.75 → clean
>   support; ratio < 0.9 with CI excluding ≥ 0.9 → refuted; else mixed.
>
> **H-05b (forgetting).** $f(\text{LoRA-16}) < f(\text{full-FT}) - \sigma_{\text{seed}}$
> on T1 (LoRA regresses less on the source domain), with the same-direction check on T2
> reported descriptively.

**Descriptive (no hypotheses):** the **quality-vs-trainable-params curve** (zero-shot →
head → LoRA-4 → LoRA-16 → full; the headline figure); LoRA-4 vs LoRA-16 (rank
sensitivity); per-recipe wall-clock and peak-VRAM (the practical PEFT sell); the
engineering claim that LoRA-wrapped UMX with B = 0 reproduces zero-shot *exactly*
(unit-tested — this is also each LoRA run's guaranteed starting point).

## 2 · Run matrix (§3.3) and the dry-run

| Domain | Recipes × seeds | Runs |
|---|---|---|
| T1 | head×1, lora4×1, **lora16×{0,1,2}**, **full×{0,1,2}** | 8 |
| T2 | head, lora4, lora16, full × 1 (seed 0) | 4 |
| LR probes (T1) | 4 recipes × 3 LRs × 500 steps | ~12 probes |

The dry-run (CPU, mock host) prints the recipe/rank/LR and the exact trainable-share
for every config — catch a mismatch before any GPU spend.

In [ ]:
# CPU-runnable now: recipe / rank / LR / trainable-share per config on the mock host.
!python ../../scripts/run_sweep.py --direction 05 --stage main --dry-run

### 2.1 · Build the domains + resolve T2 (gate G0b)

`make_domains.py` re-encodes the stems to AAC 64 kbps for T1 (deterministic ffmpeg
plan; execute needs ffmpeg) and resolves T2 by the §3.2 rule (genre subset if a
≥14-train/≥5-test cluster materializes, else pink noise at 12 dB), writing a decision
file. **RUN LATER** (needs the decoded shards).

In [ ]:
# ⚠️ RUN THIS LATER (CPU, ~1 h) — materialize the domains + resolve T2.
# python ../../scripts/make_domains.py --domain t1_aac64 --shards $SHARD_ROOT --execute
# python ../../scripts/make_domains.py --resolve-t2 --shards $SHARD_ROOT   # writes results/t2_decision.json
print('Domain build + T2 resolution — RUN LATER (needs stereo shards; ffmpeg for T1).')

## 3 · G1 sanity, LR probes, and the 12 runs

**G1 sanity** recomputes the trainable-share table against the real weights and checks
zero-shot beats do-nothing by >3 dB on the standard val split (and records the T1-val
gap for the H-05a precondition). **LR probes** pick each recipe's LR on a 500-step T1
grid (§3.4), frozen at G2. Then the **12 fine-tune runs** launch (resumable).

In [ ]:
# CPU-runnable now: the recipe trainable-share table (the table part of G1 sanity).
from singnet.peft import sanity
table = sanity(mock=True)   # RUN LATER with mock=False for the zero-shot val SI-SDR

In [ ]:
# ⚠️ RUN THIS LATER (GPU minutes) — G1 checkpoint sanity on the REAL umxhq weights.
# python -m singnet.peft.finetune_umx --sanity   # loads umxhq, recomputes shares,
#   zero-shot val SI-SDR on standard + T1 val (must beat do-nothing by > 3 dB on standard)
print('G1 real-weight sanity — RUN LATER (downloads umxhq; needs val shards).')

In [ ]:
# ⚠️ RUN THIS LATER (GPU ~1.5–2 h) — the LR probes (4 recipes × 3 LRs × 500 steps).
# python ../../scripts/run_sweep.py --direction 05 --stage probes
# -> writes probes_<recipe>_lr<lr>.yaml, runs them, logs val loss; freeze LRs into the
#    main configs before the 12 runs (gate G2).
print('LR probes — RUN LATER (GPU).')

In [ ]:
# ⚠️ RUN THIS LATER (GPU ~12–18 h total) — the 12 fine-tune runs (8 T1 + 4 T2).
# Resumable: rows already complete in the registry are skipped (a Colab disconnect
# costs minutes). Each run: 6k steps, batch 16 × 6-s stereo chunks, MSE-on-magnitude.
# python ../../scripts/run_sweep.py --direction 05 --stage main
print('12 fine-tune runs — RUN LATER (GPU; resumable).')

## 4 · The single consolidated test session (gate G3)

One pass, one CSV: every recipe checkpoint **plus zero-shot** scored on (a) the
domain's transformed 50-track test set (adapted gain) and (b) the standard test set
(forgetting). Run **only after** all LRs and checkpoints freeze on validation — no
tuning follows it (§5, §7 G3).

In [ ]:
# ⚠️ RUN THIS LATER (GPU ~1–1.5 h) — the ONE test pass for this direction.
# python ../../scripts/evaluate.py --direction 05 --test-matrix \
#     --shard-root $SHARD_ROOT --splits-csv ../../01-loss-function-study/configs/splits.csv \
#     --output-dir ../results
# -> writes results/test_matrix.csv (run_id, domain, recipe, rank, seed, eval_set, SI-SDR…)
print('Consolidated test session — RUN LATER (GPU; needs the 12 checkpoints).')

## 5 · Analysis — the quality-vs-trainable-params curve (headline)

The headline figure: mean adapted-domain vocals SI-SDR gain vs trained-parameter count
(log x-axis), zero-shot → head → lora4 → lora16 → full, per domain, with ±σ_seed on
the 3-seed cells. H-05a is the vertical gap between lora16 and full at ~4.85% vs 100%
of the host. Registry-driven; **RUN LATER** once `test_matrix.csv` exists.

In [ ]:
# ⚠️ RUN THIS LATER (CPU, needs results/test_matrix.csv) — the quality-vs-params curve.
# import pandas as pd, matplotlib.pyplot as plt
# tm = pd.read_csv('../results/test_matrix.csv')
# adapted = tm[tm['eval_set'] == 'adapted']
# # x = trainable_params (from the registry), y = mean SI-SDR gain over zero-shot, per recipe.
# ... plt.xscale('log'); plt.xlabel('trained parameters'); plt.ylabel('adapted-domain SI-SDR gain (dB)')
print('Quality-vs-trainable-params curve — RUN LATER (needs the test matrix).')

## 6 · Forgetting table + paired deltas (H-05b)

Forgetting $f(R)$ = zero-shot minus R on the **standard** test set (positive =
regression), anchored to zero-shot (the same host under our harness; the LoRA $B=0$
start *is* zero-shot). H-05b: $f(\text{lora16}) < f(\text{full}) - \sigma_{\text{seed}}$
on T1, with per-track paired deltas + bootstrap CIs for lora16-vs-full.

In [ ]:
# ⚠️ RUN THIS LATER (CPU, needs results/test_matrix.csv) — forgetting per recipe +
# the lora16-vs-full paired delta with a bootstrap CI (THEORY §6.2).
# tm = pd.read_csv('../results/test_matrix.csv'); std = tm[tm['eval_set'] == 'standard']
# f(R) = zeroshot_sisdr - R_sisdr on the standard set; tabulate per recipe + sigma_seed.
print('Forgetting table + paired deltas — RUN LATER (needs the test matrix).')

## 7 · Rank sensitivity (lora4 vs lora16)

Descriptive: does doubling-and-then-some the rank (4→16, +3.6 pp of host params) buy
gain? If lora4 already matches lora16, the adaptation is *very* low-rank (a strong PEFT
story); if lora16 ≫ lora4, capacity matters and higher ranks are worth proposing.

In [ ]:
# ⚠️ RUN THIS LATER (CPU, needs the test matrix) — lora4 vs lora16 adapted gain.
# compare adapted-gain(lora4) vs adapted-gain(lora16) on T1 (and T2), noting the
# 1.27% vs 4.85% host-param cost; a flat curve => the adaptation is essentially rank-4.
print('Rank sensitivity — RUN LATER (needs the test matrix).')

## 8 · Wall-clock / VRAM table + the with-Wiener line

The practical PEFT sell: per-recipe wall-clock and **peak VRAM** (registry columns
`wall_clock_h`, `peak_vram_gb`) — LoRA's memory win comes from optimizer state on
~5% of the params. One **with-Wiener** SI-SDR line is reported once, descriptively
(§1.3), as a constant offset on top of the no-Wiener recipe comparison.

In [ ]:
# ⚠️ RUN THIS LATER (CPU, needs results/registry.csv) — wall-clock + peak-VRAM per recipe.
# reg = pd.read_csv('../results/registry.csv')
# reg.groupby('recipe')[['wall_clock_h', 'peak_vram_gb', 'trainable_share']].mean()
print('Wall-clock / VRAM table — RUN LATER (needs the completed registry).')

## 9 · Interpretation — pre-written branches (§12; select one when results exist)

The outcome→reading→consequence matrix is pre-registered (MASTER_PLAN §12). Exactly one
branch below is selected when the test matrix exists; because the cells are named in
advance, selecting the one that occurs is *reporting*, not p-hacking.

### If **H-05a + H-05b supported**
The low-rank adaptation story transfers to recurrent separators; PEFT is the right
default for per-domain separator adaptation. → StemCraft gains a **per-library adapter**
product story at ~1–5% weight cost; write up as the headline.

### If **H-05a supported, H-05b refuted**
LoRA is efficient but not forgetting-protective here. → Adapters still win on cost; the
forgetting claim is dropped and discussed against 2602.00084's noise-robust-PEFT theory.

### If **H-05a refuted (ratio < 0.9)**
Separation adaptation is **not** low-rank at this scale — a genuine negative for the
"PEFT transfers everywhere" narrative. → Report prominently with the rank-4/16 curve;
propose (do not run) higher ranks / per-gate LSTM wrapping (THEORY §3.2).

### If **precondition failed on both domains**
`umxhq` is robust to realistic consumer shifts zero-shot. → Reframe: the practical
answer is "you don't need adaptation" — a useful product finding, tied to the Bake-Off
metric discussion.

### If **head ≥ LoRA at equal budget**, or **LSTM-LoRA trains poorly vs fc-only**
- head ≥ LoRA → the adaptation lives in the output remapping, not the representation;
  propose a per-layer ablation as future work.
- LSTM-LoRA unstable vs fc-only → recurrent LoRA is the hard part (a real engineering
  finding for the PEFT-literature gap); document the failure modes and recommend
  fc-only LoRA.

## 10 · Conclusions — what the verdict changes

- **For StemCraft:** if LoRA holds, per-user/per-library **adapters** become a product
  feature — ship one ~1–5%-size delta per domain (low-bitrate library, a genre, a
  venue) instead of a full model, with the source domain protected by lower forgetting.
- **For the PEFT literature:** this is the first careful small-scale LoRA-for-MSS
  result (gap verified 2026-07-13) — positive *or* negative, the recurrent-LoRA plumbing
  and the quality-vs-params curve are the contribution.
- **For Direction 06 (noisy labels):** 2602.00084 argues LoRA is inherently
  noise-robust; the AAC/genre shift here is a bridge to that cross-cut — if LoRA both
  adapts efficiently *and* resists the domain's label degradation, the two directions
  reinforce.
- **The engineering guarantee** already banked (no GPU): LoRA-wrapped `umxhq` with
  $B=0$ reproduces zero-shot bit-exactly, and merges back losslessly — the adapters are
  correct before a single training step.